# Setup and Import Library

In [2]:
import os
import sys
import json
import numpy as np
import pickle
from gensim.models import Word2Vec

sys.path.append(os.path.abspath('..'))
from src.preprocessing import IRPreprocessor

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Load Data

In [3]:
# Load documents
with open(os.path.join(RAW_DIR, 'cran_docs.json'), 'r') as f:
    docs = json.load(f)
    
# Load queries
with open(os.path.join(RAW_DIR, 'cran_queries.json'), 'r') as f:
    queries = json.load(f)

print(f"Loaded {len(docs)} documents and {len(queries)} queries.")

# Extract just the text content
doc_texts = [str(doc.get('body', '')) for doc in docs]
query_texts = [str(query.get('query', '')) for query in queries]
all_texts = doc_texts + query_texts

Loaded 1400 documents and 226 queries.


# Tokenization and Sequence Padding

In [4]:
# Initialize preprocessor
preprocessor = IRPreprocessor(max_seq_length=200, max_vocab_size=15000)
preprocessor.fit(all_texts)

# Vectorize documents and queries
doc_sequences = preprocessor.vectorize_and_pad(doc_texts)
query_sequences = preprocessor.vectorize_and_pad(query_texts)

print(f"Document sequences shape: {doc_sequences.shape}")
print(f"Query sequences shape: {query_sequences.shape}")

np.save(os.path.join(PROCESSED_DIR, 'doc_sequences.npy'), doc_sequences)
np.save(os.path.join(PROCESSED_DIR, 'query_sequences.npy'), query_sequences)

with open(os.path.join(PROCESSED_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(preprocessor.tokenizer, f)

Document sequences shape: (1400, 200)
Query sequences shape: (226, 200)


# Training Custom Word2Vec Embeddings

In [5]:
# Tokenize sentences for Gensim Word2Vec
tokenized_sentences = [preprocessor.tokenize_for_bm25(text) for text in all_texts]

print("Training Word2Vec model on Cranfield corpus...")
w2v_model = Word2Vec(sentences=tokenized_sentences, vector_size=100, window=5, min_count=1, workers=4)

# Create the embedding matrix for the Keras Embedding layer
vocab_size = len(preprocessor.tokenizer.word_index) + 1
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in preprocessor.tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

np.save(os.path.join(PROCESSED_DIR, 'embedding_matrix.npy'), embedding_matrix)
print("Embedding matrix saved successfully. data/processed/ is now populated!")

Training Word2Vec model on Cranfield corpus...
Embedding matrix saved successfully. data/processed/ is now populated!
